# Huấn luyện Faster R-CNN cho nhận diện biển báo (Zalo AI 2020)
Notebook này được thiết kế để chạy trên **Google Colab** (GPU T4). Nó sử dụng PyTorch thuần để xây dựng DataLoader và vòng lặp huấn luyện.

In [ ]:
# Tải bộ dữ liệu Zalo AI từ Kaggle về Google Colab bằng API (Yêu cầu phải upload file kaggle.json lên Colab trước)
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d phhasian0710/za-traffic-2020
!unzip -q -n za-traffic-2020.zip -d /content/dataset

In [ ]:
# Tải thư viện cần thiết
import os
import json
import torch
import torch.utils.data
from PIL import Image
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F

# Kiểm tra xem GPU có sẵn sàng không
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Đang sử dụng thiết bị: {device}")

In [ ]:
# Khởi tạo lớp Đọc Dữ liệu (Dataset Class)
# Lớp này sẽ đọc thẳng file COCO JSON mà không cần phải convert rườm rà qua chuẩn khác
class ZaloTrafficDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, json_file):
        self.root_dir = root_dir
        with open(json_file, 'r', encoding='utf-8') as f:
            self.coco_data = json.load(f)
        
        # Tạo danh sách tra cứu ảnh theo id
        self.images = {img['id']: img for img in self.coco_data['images']}
        
        # Gom các nhãn lại theo từng ảnh
        self.image_to_annotations = {}
        for ann in self.coco_data['annotations']:
            img_id = ann['image_id']
            if img_id not in self.image_to_annotations:
                self.image_to_annotations[img_id] = []
            self.image_to_annotations[img_id].append(ann)
            
        self.image_ids = list(self.image_to_annotations.keys())

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.images[img_id]
        
        # Đường dẫn tới file ảnh thật
        img_path = os.path.join(self.root_dir, img_info['file_name'])
        img = Image.open(img_path).convert("RGB")
        
        # Chuyển đổi nhãn (annotations) của bức ảnh này
        annos = self.image_to_annotations.get(img_id, [])
        boxes = []
        labels = []
        
        for anno in annos:
            x_min, y_min, w, h = anno['bbox']
            x_max = x_min + w
            y_max = y_min + h
            boxes.append([x_min, y_min, x_max, y_max])
            
            # Faster R-CNN bắt buộc class nền (background) phải là 0
            # Do đó id của biển báo từ Zalo (1-7) được giữ nguyên, không cần trừ đi 1 như YOLO
            labels.append(anno['category_id'])
            
        # Chuyển danh sách thành tensor của PyTorch để đưa vào tính toán
        if len(boxes) > 0:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
        else:
            # Xử lý trường hợp ảnh không có biển báo nào (tránh lỗi)
            boxes = torch.empty((0, 4), dtype=torch.float32)
            labels = torch.empty((0,), dtype=torch.int64)
        
        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["image_id"] = torch.tensor([img_id])
        
        # Biến đổi ảnh thành tensor hình ảnh chuẩn
        img_tensor = F.to_tensor(img)
        
        return img_tensor, target

    def __len__(self):
        return len(self.image_ids)

In [ ]:
# Hàm tạo mô hình Faster R-CNN
def get_model(num_classes):
    # Tải mô hình đã được huấn luyện sẵn trên bộ dữ liệu COCO (pretrained=True giúp học cực nhanh)
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
    
    # Lấy số chiều của đầu vào ở lớp dự đoán (classifier) cuối cùng
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    
    # Thay cái đầu dự đoán cũ (91 classes) bằng một cái đầu dự đoán mới của ta
    # Bộ Zalo AI có 7 loại biển báo + 1 loại nền (background) => num_classes = 8
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    return model

In [ ]:
# Khai báo đường dẫn dữ liệu (Bạn nhớ sửa theo đường dẫn thực tế trên Colab của bạn)
# Ví dụ: root_dir = '/content/dataset/za_traffic_2020/traffic_train/images'
root_dir = '/content/dataset/za_traffic_2020/traffic_train/images'
json_file = '/content/dataset/za_traffic_2020/traffic_train/train_traffic_sign_dataset.json'

# Khởi tạo Dataset và DataLoader để nhồi dữ liệu (Batch) vào máy tính
def collate_fn(batch):
    return tuple(zip(*batch))

try:
    dataset = ZaloTrafficDataset(root_dir, json_file)
    data_loader = torch.utils.data.DataLoader(
        dataset, batch_size=4, shuffle=True, collate_fn=collate_fn
    )
    print(f"Đã tạo xong DataLoader với tổng cộng {len(dataset)} bức ảnh hợp lệ.")
except Exception as e:
    print("Vui lòng tải dữ liệu về thư mục Colab trước khi khởi tạo DataLoader!")
    print("Lỗi:", e)

In [ ]:
# Gọi thư viện hiển thị thanh tiến trình và kết nối Drive
from tqdm import tqdm
from google.colab import drive
import os

# Yêu cầu quyền truy cập Google Drive để lưu file vĩnh viễn
drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/DoAn_NhanDienBienBao/faster_rcnn_highres'
os.makedirs(save_dir, exist_ok=True) # Tự động tạo thư mục nếu chưa có

# Chuẩn bị huấn luyện
num_classes = 8  
model = get_model(num_classes)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

num_epochs = 10 
print("Bắt đầu huấn luyện mô hình...")

# Vòng lặp huấn luyện có thanh tiến trình
for epoch in range(num_epochs):
    model.train()  
    epoch_loss = 0
    
    # Bọc data_loader bằng tqdm để xem phần trăm chạy
    progress_bar = tqdm(data_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for images, targets in progress_bar:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()
        
        # Cập nhật độ lỗi liên tục lên thanh tiến trình
        progress_bar.set_postfix(loss=losses.item())

    print(f"\nHoàn thành Epoch {epoch+1} - Tổng độ lỗi (Loss): {epoch_loss:.4f}")
    
    # RỦI RO ĐÃ ĐƯỢC KHẮC PHỤC: Lưu an toàn vào Google Drive NGAY LẬP TỨC sau mỗi Epoch
    save_path = os.path.join(save_dir, 'faster_rcnn_last.pth')
    torch.save(model.state_dict(), save_path)
    print(f"[Bảo mật] Đã tự động lưu checkpoint của Epoch {epoch+1} vào Drive.")

# Lưu thẳng vào Google Drive của bạn (File hoàn thiện cuối cùng)
final_path = os.path.join(save_dir, 'faster_rcnn_best.pth')
torch.save(model.state_dict(), final_path)
print(f"Đã lưu mô hình vĩnh viễn và an toàn tại: {final_path}")